# I-Turn — Colab prototype

Open conversation with a local Qwen2.5-3B, DASS-21 and Lifestyle R3 underneath.
Nothing leaves this runtime.

Runtime → Change runtime type → **T4 GPU** before running anything.

The logic lives in the `app/` package, not in this notebook. The notebook is a
shell — the same code runs unchanged on the RTX 5050 via FastAPI. If you want
to change how a session behaves, edit `app/session.py`, not a cell here.

## 1. Setup

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes gradio

# Upload the iturn/ folder (or clone it), then:
import sys, os
sys.path.insert(0, '/content/iturn')
os.environ['ITURN_BACKEND'] = 'transformers'
os.environ['ITURN_HF_MODEL'] = 'Qwen/Qwen2.5-3B-Instruct'
print('ready')

## 2. Load the model

4-bit so it fits a T4 with room to spare. `bnb_4bit_compute_dtype=float16` on
T4 — note the local 5050 path uses bfloat16 instead, which T4 doesn't support.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = 'Qwen/Qwen2.5-3B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

if torch.cuda.is_available():
    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, device_map='auto', quantization_config=quant)
    print('loaded on', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('No GPU. Runtime -> Change runtime type -> T4 GPU.')

# Hand the loaded model to the package so it doesn't load a second copy.
from app import llm
llm._hf.update({'tok': tokenizer, 'model': model, 'torch': torch})

## 3. Smoke test the deterministic layers

These run without the model. If any of this fails, the model won't save you.

In [ ]:
from app.instruments import dass21
from app import safety

r = dass21.score({i.number: 3 for i in dass21.ITEMS})
print(r.plain_summary(), '| referral:', r.referral_indicated)

for t in ["my phone died", "dying of boredom", "I don't want to live anymore",
          "panic attacks before vivas", "hopeless about placements"]:
    a = safety.assess(t)
    print(f'{a.level.name:8} <- {t}')

## 4. Chat

Open-ended. The questionnaire is offered only after the conversation has
actually gone somewhere, and declining it costs nothing.

In [ ]:
import gradio as gr
from app.session import Session, Mode

STATE = {}

def start(pseudonym, mode, language):
    s = Session(pseudonym=pseudonym or 'anon',
                mode=Mode.STORY if mode == 'Story' else Mode.INCOGNITO,
                language=language)
    s.start()
    STATE['s'] = s
    return [], f'Session {s.session_id} · {s.mode.value}'

def chat(message, history):
    s = STATE.get('s')
    if s is None:
        return "Press 'Begin session' first."
    out = s.turn(message)
    parts = []
    if out.get('message'):
        parts.append(out['message'])
    if out.get('offer'):
        parts.append('\n\n' + out['offer']['prompt'] +
                     "\n\n(Type 'yes' to start it, or just keep talking.)")
    if out.get('items'):
        i = out['items'][0]
        anchors = '\n'.join(f'  {k} — {v}' for k, v in i['anchors'].items())
        parts.append(f"\n\n**{i['number']}/21 · over the past week**\n"
                     f"{i['text']}\n\n{anchors}")
    return '\n'.join(parts) or '…'

def handoff():
    s = STATE.get('s')
    if s is None:
        return 'No active session.'
    import json
    return json.dumps(s.handoff(), indent=2)

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown('### I-Turn')
    with gr.Row():
        pseudonym = gr.Textbox(label='Name to go by', value='U_83F91A')
        mode = gr.Radio(['Incognito', 'Story'], value='Incognito', label='Memory')
        language = gr.Dropdown(['English', 'Hindi', 'Kannada', 'Tamil'],
                               value='English', label='Language')
    status = gr.Markdown('')
    begin = gr.Button('Begin session', variant='primary')
    chatbot = gr.ChatInterface(fn=chat, type='messages')
    with gr.Accordion('Counsellor handoff (would be auth-gated in production)', open=False):
        btn = gr.Button('Generate')
        out = gr.Code(language='json')
        btn.click(handoff, outputs=out)
    begin.click(start, [pseudonym, mode, language], [chatbot.chatbot, status])

demo.launch(debug=True, share=True)

---

### Known gaps in this notebook

- `safety.py` patterns are English-only. The language dropdown offers four,
  so risk detection silently degrades in three of them. Restrict the dropdown
  for any real pilot, or write the patterns first.
- The DASS block is text-only here. The FastAPI UI renders proper 0–3 buttons,
  which is the correct administration — Gradio's chat box makes it easy for a
  student to answer with something the extractor has to guess at, and a guessed
  DASS response is not a DASS response.
- `/api/handoff` has no auth anywhere yet. Don't expose this notebook's share
  link to anyone you wouldn't show a counsellor's screen to.